# 12 — Robust CEM + Meta-Equilibrium + Submission Panel (CPU)

This replaces **E005 + equilibrium E007 + E006**.

It performs offline population search, creates a payoff matrix, solves a robust meta mixture, embeds the result into the tiny runtime artifact, smoke-tests the final agent, and builds a controlled submission panel.

## Inputs
- Required: output Dataset from notebook 11
- Recommended: output Dataset from notebook 10, for provenance/diagnostics
- Code repo Dataset: optional; exact recorded commit is auto-cloned when Internet is available
- Accelerator: **None**
- Internet: OFF with code Dataset, otherwise ON for auto-clone

## Default search budget
Balanced for Kaggle CPU. Increase `ITERATIONS`, `POPULATION`, and `SEEDS` only after the first full pass completes.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,shutil

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/kagv2')
WORK.mkdir(parents=True,exist_ok=True)

def find_repo():
    roots=[INPUT,Path('/kaggle/working'),Path.cwd()]
    hit=next((p for r in roots if r.exists() for p in r.rglob('src/kagv2/__init__.py')),None)
    return hit.parents[2] if hit else None

def expected_commit():
    hits=list(INPUT.rglob('repo_commit.txt')) if INPUT.exists() else []
    if hits:
        x=hits[0].read_text().strip()
        return x if x else None
    return None

ROOT=find_repo()
if ROOT is None:
    dst=Path('/kaggle/working/kaggriculture')
    if not dst.exists():
        r=subprocess.run(
            ['git','clone','--depth','1','https://github.com/sidhulyalkar/kaggriculture.git',str(dst)],
            capture_output=True,text=True
        )
        if r.returncode:
            raise RuntimeError(
                'Could not find an attached code repo and GitHub clone failed. '
                'Turn Internet ON or attach your kaggriculture-code-repo Dataset.\n'+r.stderr[-2000:]
            )
    ROOT=dst
    ref=expected_commit()
    if ref:
        subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin',ref],capture_output=True,text=True)
        c=subprocess.run(['git','-C',str(ROOT),'checkout',ref],capture_output=True,text=True)
        if c.returncode:
            print('WARN: could not checkout pinned commit',ref,c.stderr[-500:])

sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/'src'))
commit=subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
if commit:
    (WORK/'repo_commit.txt').write_text(commit)
print('ROOT =',ROOT)
print('WORK =',WORK)
print('COMMIT =',commit or 'dataset snapshot')


In [ ]:
import pandas as pd,numpy as np,json,os,copy,time,math
from joblib import Parallel,delayed
from submission.parametric_agent import ParametricMind,DEFAULT_PARAMS
from submission.base_controller import HarvestMind
from baselines.v1.counter_agent import CounterMeta,TournamentMind
from src.kagv2.simulator import Game
from src.kagv2.cem import PARAM_NAMES,DEFAULT,LOW,HIGH,decode
from src.kagv2.equilibrium import payoff_from_results,robust_population_mix

def find_input(name,required=True):
    hits=list(INPUT.rglob(name))
    if hits:return hits[0]
    if required:raise FileNotFoundError(name)
    return None

learned_path=find_input('learned_model.json')
model=json.loads(learned_path.read_text())
lib_path=find_input('macro_library.json')
lib=json.loads(lib_path.read_text()) if lib_path else {}
print('model',learned_path,'archetypes',len((model.get('archetype') or {}).get('centroids',[])))


In [ ]:
# -------- Build replay-derived macro opponents --------
# We map each mined day-indexed macro schedule into the same compact parameter
# family searched by CEM. This is approximate, but adds real ladder diversity
# beyond our hand-authored baselines.
def day_record(v,day):
    days=v.get('days',{})
    if not days:return {}
    keys=sorted(int(k) for k in days)
    k=min(keys,key=lambda x:abs(x-day))
    return days.get(str(k),{})

def macro_to_params(v):
    p=dict(DEFAULT_PARAMS)
    d3=day_record(v,3); d7=day_record(v,7); d11=day_record(v,11)
    d15=day_record(v,15); d20=day_record(v,20)
    def put(name,rec,key,lo,hi):
        if key in rec and rec[key]==rec[key]:
            p[name]=int(round(max(lo,min(hi,float(rec[key])))))
    put('hands_early',d3,'hands',1,8); put('hands_mid',d7,'hands',3,13); put('hands_late',d11,'hands',7,16)
    put('cow_mid',d7,'animal_COW',2,10); put('sheep_mid',d7,'animal_SHEEP',1,10)
    put('cow_late',d15,'animal_COW',4,12); put('sheep_late',d15,'animal_SHEEP',2,12)
    put('q1_wheat',d3,'crop_WHEAT',3,15); put('q1_melon',d3,'crop_MELON',4,18)
    put('mid_wheat',d11,'crop_WHEAT',3,25); put('mid_melon',d11,'crop_MELON',0,20)
    put('late_wheat',d20,'crop_WHEAT',8,35)
    return p

opponents=[
    ('harvest',('class','harvest')),
    ('counter_meta',('class','counter')),
    ('tournament_v1',('class','tournament')),
]
for cl,v in sorted(lib.items(),key=lambda kv:float(kv[1].get('win_rate',0)),reverse=True)[:5]:
    opponents.append((f'replay_arch_{cl}',('param',macro_to_params(v))))
print('Opponent zoo:',[x[0] for x in opponents])


In [ ]:
# -------- Parallel robust CEM --------
ITERATIONS=4
POPULATION=20
ELITE_FRAC=.25
SEEDS=list(range(4))
N_JOBS=max(1,min(4,(os.cpu_count() or 2)-1))
RNG=np.random.default_rng(20260817)

def make_opp(spec):
    kind,val=spec
    if kind=='param': return ParametricMind(val).act
    if val=='harvest': return HarvestMind().act
    if val=='counter': return CounterMeta().act
    return TournamentMind().act

def matchup(params,spec,seed,seat):
    mine=ParametricMind(params).act
    opp=make_opp(spec)
    agents=[mine,opp] if seat==0 else [opp,mine]
    cash=Game(seed=100000+seed*17+seat).run(agents)
    a,b=(cash[0],cash[1]) if seat==0 else (cash[1],cash[0])
    return 1.0 if a>b else .5 if a==b else 0.0

def candidate_vector(params):
    vals=[]
    for _,spec in opponents:
        scores=[matchup(params,spec,s,seat) for s in SEEDS for seat in (0,1)]
        vals.append(float(np.mean(scores)-.5))
    return np.asarray(vals)

def robust_score(v,ww=.30,cw=.20):
    mean=float(np.mean(v)); worst=float(np.min(v))
    k=max(1,int(math.ceil(.35*len(v))))
    cvar=float(np.mean(np.sort(v)[:k]))
    return (1-ww-cw)*mean+ww*worst+cw*cvar

mu=DEFAULT.copy();sig=(HIGH-LOW)*.20
k=max(2,int(POPULATION*ELITE_FRAC))
history=[];best=(-1e9,None,None,None)
for it in range(ITERATIONS):
    pop=np.clip(RNG.normal(mu,sig,size=(POPULATION,len(mu))),LOW,HIGH)
    params=[decode(x) for x in pop]
    vectors=Parallel(n_jobs=N_JOBS,prefer='processes')(
        delayed(candidate_vector)(p) for p in params
    )
    scores=np.array([robust_score(v) for v in vectors])
    order=np.argsort(scores)[::-1]
    elite=pop[order[:k]]
    mu=.7*mu+.3*elite.mean(0)
    sig=np.maximum(.03*(HIGH-LOW),.7*sig+.3*elite.std(0))
    bi=int(order[0])
    if scores[bi]>best[0]:
        best=(float(scores[bi]),pop[bi].copy(),params[bi],vectors[bi])
    rec={'iteration':it,'best':float(scores[bi]),'mean':float(scores.mean()),'global_best':best[0]}
    history.append(rec);print(rec)

cem_best=best[2]
(WORK/'cem_best.json').write_text(json.dumps({'best':cem_best,'history':history,'opponent_names':[x[0] for x in opponents]},indent=2))
print('BEST',cem_best,'vector',best[3])


In [ ]:
# -------- Policy zoo + payoff matrix --------
# Include default, robust-CEM, and the strongest replay-derived macro policies.
policy_params={'default':dict(DEFAULT_PARAMS),'cem_robust':dict(cem_best)}
for cl,v in sorted(lib.items(),key=lambda kv:float(kv[1].get('win_rate',0)),reverse=True)[:2]:
    policy_params[f'replay_policy_{cl}']=macro_to_params(v)
(WORK/'policy_params.json').write_text(json.dumps(policy_params,indent=2,sort_keys=True))

def eval_policy(pname,params,oname,spec,seed,seat):
    mine=ParametricMind(params).act;opp=make_opp(spec)
    agents=[mine,opp] if seat==0 else [opp,mine]
    cash=Game(seed=200000+seed*19+seat).run(agents)
    a,b=(cash[0],cash[1]) if seat==0 else (cash[1],cash[0])
    return {'policy':pname,'opponent_archetype':oname,'seed':seed,'seat':seat,
            'score':1. if a>b else .5 if a==b else 0.,'margin':a-b}

jobs=[(pn,pp,on,sp,s,seat)
      for pn,pp in policy_params.items()
      for on,sp in opponents
      for s in SEEDS for seat in (0,1)]
rows=Parallel(n_jobs=N_JOBS,prefer='processes')(
    delayed(eval_policy)(*j) for j in jobs
)
matchups=pd.DataFrame(rows)
matchups.to_parquet(WORK/'policy_matchups.parquet',index=False)
display(matchups.groupby(['policy','opponent_archetype']).score.agg(['mean','count']))


In [ ]:
# -------- Meta equilibrium --------
policies,opp_names,A,N=payoff_from_results(matchups,shrink=8.0)
counts=matchups.groupby('opponent_archetype').size().reindex(opp_names,fill_value=0).to_numpy(float)
prior=(counts/counts.sum()).tolist()
meta=robust_population_mix(A,opponent_prior=prior,equilibrium_weight=.40,iterations=12000)
meta.update({
    'policy_names':policies,
    'archetype_names':opp_names,
    'payoff':A.tolist(),
    'match_counts':N.tolist(),
    'policy_params':policy_params,
    'default_policy':'default' if 'default' in policies else policies[0],
    'min_archetype_confidence':.62,
    'switch_margin':.025,
    'prior_strength':.018,
})
(WORK/'meta_artifact.json').write_text(json.dumps(meta,indent=2,sort_keys=True))
print('META MIX',dict(zip(policies,meta['policy_mixture'])))
print('expected',meta['expected_meta_value'],'worst',meta['worst_archetype_value'],
      'duality_gap',meta['equilibrium']['duality_gap'])


In [ ]:
# -------- Embed model and build controlled leaderboard panel --------
model['version']=max(2,int(model.get('version',0)))
model['meta']=meta
model.setdefault('probe',{'enabled':False})
model.setdefault('provenance',{})
model['provenance'].update({'repo_commit':commit,'search_notebook':'12_search_meta_submit_cpu'})
promoted=WORK/'learned_model_promoted.json'
promoted.write_text(json.dumps(model,indent=2,sort_keys=True))

# Build all five artifacts. Recommended live use is S1-S4; S0 is a fresh
# variance/control check and can be skipped if daily slots are more valuable.
subprocess.run([
    sys.executable,str(ROOT/'scripts'/'build_experiment_submissions.py'),
    '--model',str(promoted),
    '--meta',str(WORK/'meta_artifact.json'),
    '--out-dir',str(WORK/'experiments')
],check=True)

manifest=json.loads((WORK/'experiments'/'experiment_manifest.json').read_text())
display(pd.DataFrame(manifest)[['variant','description','bytes','sha256']])


In [ ]:
# -------- Submission package + runtime smoke test --------
# Also materialize the full promoted model into the repo's submission folder for
# the standard single-candidate builder, then restore the original afterward.
import shutil,tarfile,py_compile
target=ROOT/'submission'/'learned_model.json'
backup=target.read_text() if target.exists() else None
try:
    target.write_text(promoted.read_text())
    files=['main.py','predictive_agent.py','parametric_agent.py','base_controller.py','runtime_model.py','meta_runtime.py']
    for f in files: py_compile.compile(str(ROOT/'submission'/f),doraise=True)
    from submission.predictive_agent import PredictiveMind
    for seed in range(3):
        cash=Game(seed=700000+seed).run([PredictiveMind().act,TournamentMind().act])
        print('full-v2 smoke',seed,cash)
finally:
    if backup is not None: target.write_text(backup)

summary={
    'repo_commit':commit,
    'cem_best_score':best[0],
    'policy_count':len(policy_params),
    'opponent_count':len(opponents),
    'meta_expected_value':meta['expected_meta_value'],
    'meta_worst_value':meta['worst_archetype_value'],
    'equilibrium_gap':meta['equilibrium']['duality_gap'],
    'variants':[x['variant'] for x in manifest],
}
(WORK/'final_search_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


# Recommended next ladder submissions

Use the existing V1 as the historical control. For the **next four slots**, submit in this order:

1. **S1_robust_fixed** — tests whether offline macro search itself is the edge.
2. **S2_market_only** — isolates future-supply/predictive selling.
3. **S3_meta_only** — isolates opponent belief + policy selection.
4. **S4_full** — tests interaction between the two learned components.

Keep the fifth slot reserved for a packaging/runtime fix or a refined winner after early hosted replays.

If you want a fresh control to quantify rating/matchmaking noise, submit `S0_control`, but I would normally preserve that slot because the existing V1 already provides a control lineage.

Do not rank variants from one or two games. Compare hosted replays, both-seat behavior where available, opponent strength, and component-specific failure modes.
